In [13]:
# 1 Carga de Librerias
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA

In [14]:
# 2. Leer el archivo CSV
EmpleadosAttrition = pd.read_csv('empleadosRETO.csv')

In [15]:
# 3. Eliminar columnas que no aportan información (constantes o IDs)
columnas_borrar = ['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours']
EmpleadosAttrition.drop(columns=columnas_borrar, inplace=True)

In [16]:
# 4, 5 y 6. Ingeniería de fechas: Calcular años en la compañía (hasta 2018)
# Convertimos HiringDate a formato fecha para extraer el año
EmpleadosAttrition['HiringDate'] = pd.to_datetime(EmpleadosAttrition['HiringDate'], errors='coerce', format='mixed', dayfirst=True)
EmpleadosAttrition['Year'] = EmpleadosAttrition['HiringDate'].dt.year.fillna(0).astype(int)
EmpleadosAttrition['YearsAtCompany'] = 2018 - EmpleadosAttrition['Year']

In [17]:
# 7, 8 y 9. Limpiar DistanceFromHome (quitar "km" y convertir a entero)
EmpleadosAttrition.rename(columns={'DistanceFromHome': 'DistanceFromHome_km'}, inplace=True)
# Extraemos solo los dígitos y convertimos a entero
EmpleadosAttrition['DistanceFromHome'] = EmpleadosAttrition['DistanceFromHome_km'].str.extract('(\+?\d+)').astype(int)

<>:4: SyntaxWarning: invalid escape sequence '\+'
<>:4: SyntaxWarning: invalid escape sequence '\+'
/tmp/ipython-input-2891056724.py:4: SyntaxWarning: invalid escape sequence '\+'
  EmpleadosAttrition['DistanceFromHome'] = EmpleadosAttrition['DistanceFromHome_km'].str.extract('(\+?\d+)').astype(int)


In [18]:
# 10. Borrar columnas temporales
EmpleadosAttrition.drop(columns=['Year', 'HiringDate', 'DistanceFromHome_km'], inplace=True)

In [19]:
# 11. Sueldo promedio por departamento (Tabla informativa)
SueldoPromedioDepto = EmpleadosAttrition.groupby('Department')[['MonthlyIncome']].mean()
SueldoPromedio = SueldoPromedioDepto # Variable solicitada
print("Sueldo Promedio por Departamento:\n", SueldoPromedio)

Sueldo Promedio por Departamento:
                         MonthlyIncome
Department                           
Human Resources           6239.888889
Research & Development    6804.149813
Sales                     7188.250000


In [20]:
# 12. Escalar MonthlyIncome entre 0 y 1
scaler = MinMaxScaler()
EmpleadosAttrition['MonthlyIncome'] = scaler.fit_transform(EmpleadosAttrition[['MonthlyIncome']])

In [21]:
# 13. Convertir variables categóricas a numéricas (Label Encoding manual/mapeo)
# Usaremos factorize para convertir texto a números rápidamente
categoricas = ['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'Attrition']
for col in categoricas:
    EmpleadosAttrition[col] = pd.factorize(EmpleadosAttrition[col])[0]

In [26]:
# 16 y 17. PCA (Análisis de Componentes Principales)
# Primero extraemos las características (X) sin la variable de salida (Attrition)
X = EmpleadosAttritionFinal.drop(columns=['Attrition'])

# Aplicamos PCA buscando explicar el 80% de la varianza
pca_model = PCA(n_components=0.80)
EmpleadosAttritionPCA = pca_model.fit_transform(X)

# Agregar los componentes al frame final
for i in range(EmpleadosAttritionPCA.shape[1]):
    col_name = f'C{i}'
    EmpleadosAttritionFinal = EmpleadosAttritionFinal.assign(**{col_name: EmpleadosAttritionPCA[:, i]})

In [27]:
# 18. Guardar el archivo final
EmpleadosAttritionFinal.to_csv('EmpleadosAttritionFinal.csv', index=False)

print("\n¡Proceso completado! El archivo 'EmpleadosAttritionFinal.csv' ha sido generado.")


¡Proceso completado! El archivo 'EmpleadosAttritionFinal.csv' ha sido generado.
